In [18]:
import pandas as pd

# Read the TSV file with specific columns
columns_to_read = ['rowid', 'Scan', 'Annotation', 'AnnotationOther', 
                   'MinNTermAdd', 'minNTermSubtract', 'MinCTermAdd', 'minCTermSubtract']
mismatch_df = pd.read_csv('PRISMAL_REPROCESS_PA_UniproKB_Normal123.tsv', sep='\t', usecols=columns_to_read)

# Display basic information about the dataset
print(f"Dataset shape: {mismatch_df.shape}")
print(f"Columns: {list(mismatch_df.columns)}")
print("\nFirst few rows:")
mismatch_df.head()

Dataset shape: (11224, 8)
Columns: ['rowid', 'Scan', 'Annotation', 'AnnotationOther', 'MinNTermAdd', 'minNTermSubtract', 'MinCTermAdd', 'minCTermSubtract']

First few rows:


,rowid,Scan,Annotation,AnnotationOther,MinNTermAdd,minNTermSubtract,MinCTermAdd,minCTermSubtract
0,1,6352,+42.011FRNFGGLLGPMDEPVGMQKWGK,KQVEVDAQQC+57.021MLEILDTAGTEQ,0,0,5,21
1,2,6353,ADEEFQ+0.984ILANSWRYSSAFTNR,KQVEVDAQQC+57.021MLEILDTAGTEQ,0,0,5,21
2,3,6363,ADEEFQ+0.984ILANSWRYSSAFTNR,KQVEVDAQQC+57.021MLEILDTAGTEQ,0,0,5,21
3,4,6351,N+0.984VPDSLGFLQNFSPLSVHRDEM,KQVEVDAQQC+57.021MLEILDTAGTEQ,0,0,5,21
4,5,21431,LSLQLTDELYPGLYK,DSIQLHAKSFVSNHTA,4,8,8,8


create a new tsv named mismatch_summary.tsv with the following columns: peptide, peptide_demod, reason_mismatch,'MinNTermAdd', 'minNTermSubtract', 'MinCTermAdd', 'minCTermSubtract' where go through each column of mismatch_df, the peptide in new csv is  AnnotationOther, and peptide_demod is AnnotationOther but get rid of the modifications, for example "+42.011FRNF+28.011GGL" is FRNFGGLLGPMDEPVGMQKWGK. and 'MinNTermAdd', 'minNTermSubtract', 'MinCTermAdd', 'minCTermSubtract' columns are the same from mismatch df. leave reason_mismatch empty for now

In [19]:
import re

def remove_modifications(peptide_string):
    """Remove modification annotations from peptide sequence"""
    if pd.isna(peptide_string):
        return ''
    # Remove modification patterns like +42.011, +28.011, etc.
    cleaned = re.sub(r'\+[\d.]+', '', str(peptide_string))
    return cleaned

# Create the new dataframe with required columns
summary_df = pd.DataFrame({
    'scan': mismatch_df['Scan'],
    'peptide': mismatch_df['AnnotationOther'],
    'peptide_demod': mismatch_df['AnnotationOther'].apply(remove_modifications),
    'peptide_length': mismatch_df['AnnotationOther'].apply(remove_modifications).apply(len),
    'peptide_type':'',
    'reason_mismatch': '',  # Empty for now as requested
    'MinNTermAdd': mismatch_df['MinNTermAdd'],
    'minNTermSubtract': mismatch_df['minNTermSubtract'],
    'MinCTermAdd': mismatch_df['MinCTermAdd'],
    'minCTermSubtract': mismatch_df['minCTermSubtract']
})

# Save to TSV file
summary_df.to_csv('mismatch_summary.tsv', sep='\t', index=False)

print(f"Created mismatch_summary.tsv with {len(summary_df)} rows")
print("\nFirst few rows of the summary:")
summary_df.head()

Created mismatch_summary.tsv with 11224 rows

First few rows of the summary:


,scan,peptide,peptide_demod,peptide_length,peptide_type,reason_mismatch,MinNTermAdd,minNTermSubtract,MinCTermAdd,minCTermSubtract
0,6352,KQVEVDAQQC+57.021MLEILDTAGTEQ,KQVEVDAQQCMLEILDTAGTEQ,22,,,0,0,5,21
1,6353,KQVEVDAQQC+57.021MLEILDTAGTEQ,KQVEVDAQQCMLEILDTAGTEQ,22,,,0,0,5,21
2,6363,KQVEVDAQQC+57.021MLEILDTAGTEQ,KQVEVDAQQCMLEILDTAGTEQ,22,,,0,0,5,21
3,6351,KQVEVDAQQC+57.021MLEILDTAGTEQ,KQVEVDAQQCMLEILDTAGTEQ,22,,,0,0,5,21
4,21431,DSIQLHAKSFVSNHTA,DSIQLHAKSFVSNHTA,16,,,4,8,8,8


def a fucntion for checking reason mismatch, and filling peptide type,  default is leave blank for mismatch_reason
    miss_3_cleavages ----Peptide has 3+ missed cleavages for trypsin
    lost_3_aa_Nterm--- Peptide lost 3+ AAs from the N-term (prefix) - this is the MinNTermAdd if its larger than 3
    Cterm_non_tryptic --- Peptide C-term (suffix/end) is non-Tryptic (i.e., no K/R at the end)

    for peptide_type column 
        read from peptide_length
        HLA1 -- Peptide length 8-12 = mark as HLA1
        HLA2 -- Peptide length 9-20 = mark as HLA2
        Otherwise mark as OtherLength


In [20]:
def analyze_peptide_mismatches(df):
    """
    Function to analyze peptide mismatches and assign peptide types
    """
    def count_missed_cleavages(peptide_seq):
        """Count missed cleavages for trypsin (K/R not at C-terminus)"""
        if pd.isna(peptide_seq) or len(peptide_seq) == 0:
            return 0
        # Count K and R that are not at the end of the sequence
        missed = 0
        for i, aa in enumerate(peptide_seq[:-1]):  # Exclude last position
            if aa in ['K', 'R']:
                missed += 1
        return missed
    
    def is_cterm_tryptic(peptide_seq):
        """Check if C-terminus is tryptic (ends with K or R)"""
        if pd.isna(peptide_seq) or len(peptide_seq) == 0:
            return False
        return peptide_seq[-1] in ['K', 'R']
    
    def get_peptide_type(length):
        """Assign peptide type based on length"""
        if 8 <= length <= 12:
            return 'HLA1'
        elif 9 <= length <= 20:
            return 'HLA2'
        else:
            return 'OtherLength'
    
    def get_mismatch_reason(row):
        """Determine mismatch reason based on criteria"""
        reasons = []
        
        # Check for 3+ missed cleavages
        missed_cleavages = count_missed_cleavages(row['peptide_demod'])
        if missed_cleavages >= 3:
            reasons.append('miss_3_cleavages')
        
        # Check for lost 3+ AAs from N-term
        if row['MinNTermAdd'] >= 3:
            reasons.append('lost_3_aa_Nterm')
        
        # Check for non-tryptic C-term
        if not is_cterm_tryptic(row['peptide_demod']):
            reasons.append('Cterm_non_tryptic')

        if row['peptide_length'] > 40:
            reasons.append('peptide_length_>_40')
        
        # Check for multiple modifications (more than one + sign in peptide)
        # Check for multiple modifications (count both + and - signs)
        mod_count = row['peptide'].count('+') + row['peptide'].count('-')
        if mod_count > 1:
            reasons.append('multiple_modifications')

        # Check for non-standard modifications
        standard_mods = [1, 16, 42, 43, -17]
        tolerance = 0.5  # Allow for rounding
        
        # Extract all modifications from peptide string
        mod_matches = re.findall(r'([+-][\d.]+)', str(row['peptide']))
        for mod_str in mod_matches:
            mod_value = float(mod_str)
            is_standard = any(abs(mod_value - std_mod) <= tolerance for std_mod in standard_mods)
            if not is_standard:
                reasons.append('non_standard_modification')
                break
        
        return '; '.join(reasons) if reasons else ''
    
    # Apply the functions to update the dataframe
    df['peptide_type'] = df['peptide_length'].apply(get_peptide_type)
    df['reason_mismatch'] = df.apply(get_mismatch_reason, axis=1)
    
    return df

# Apply the analysis to the summary dataframe
summary_df = analyze_peptide_mismatches(summary_df)

# Save the updated dataframe
summary_df.to_csv('mismatch_summary.tsv', sep='\t', index=False)

print(f"Updated mismatch_summary.tsv with peptide types and mismatch reasons")
print("\nPeptide type distribution:")
print(summary_df['peptide_type'].value_counts())
print("\nMismatch reason distribution:")
print(summary_df['reason_mismatch'].value_counts())
print("\nFirst few rows:")
summary_df.head()

Updated mismatch_summary.tsv with peptide types and mismatch reasons

Peptide type distribution:
HLA2           5218
OtherLength    3700
HLA1           2306
Name: peptide_type, dtype: int64

Mismatch reason distribution:
lost_3_aa_Nterm; Cterm_non_tryptic                                                                         3147
miss_3_cleavages; Cterm_non_tryptic                                                                        2029
Cterm_non_tryptic                                                                                          1354
multiple_modifications; non_standard_modification                                                           612
miss_3_cleavages                                                                                            544
lost_3_aa_Nterm                                                                                             502
lost_3_aa_Nterm; Cterm_non_tryptic; multiple_modifications                                                 

,scan,peptide,peptide_demod,peptide_length,peptide_type,reason_mismatch,MinNTermAdd,minNTermSubtract,MinCTermAdd,minCTermSubtract
0,6352,KQVEVDAQQC+57.021MLEILDTAGTEQ,KQVEVDAQQCMLEILDTAGTEQ,22,OtherLength,Cterm_non_tryptic; non_standard_modification,0,0,5,21
1,6353,KQVEVDAQQC+57.021MLEILDTAGTEQ,KQVEVDAQQCMLEILDTAGTEQ,22,OtherLength,Cterm_non_tryptic; non_standard_modification,0,0,5,21
2,6363,KQVEVDAQQC+57.021MLEILDTAGTEQ,KQVEVDAQQCMLEILDTAGTEQ,22,OtherLength,Cterm_non_tryptic; non_standard_modification,0,0,5,21
3,6351,KQVEVDAQQC+57.021MLEILDTAGTEQ,KQVEVDAQQCMLEILDTAGTEQ,22,OtherLength,Cterm_non_tryptic; non_standard_modification,0,0,5,21
4,21431,DSIQLHAKSFVSNHTA,DSIQLHAKSFVSNHTA,16,HLA2,lost_3_aa_Nterm; Cterm_non_tryptic,4,8,8,8


In [21]:
# Filter for rows where reason_mismatch is blank and peptide_type is OtherLength
filtered_df = summary_df[(summary_df['reason_mismatch'] == '') & (summary_df['peptide_type'] == 'OtherLength')]

# Save the filtered data to a new TSV file
filtered_df.to_csv('not_expected_mismatch.tsv', sep='\t', index=False)

print(f"Created filtered_otherlength_nomismatch.tsv with {len(filtered_df)} rows")
print(f"Total rows in original data: {len(summary_df)}")
print(f"Filtered rows (blank reason_mismatch AND OtherLength): {len(filtered_df)}")
print("\nFirst few rows of filtered data:")
filtered_df.head()

Created filtered_otherlength_nomismatch.tsv with 35 rows
Total rows in original data: 11224
Filtered rows (blank reason_mismatch AND OtherLength): 35

First few rows of filtered data:


,scan,peptide,peptide_demod,peptide_length,peptide_type,reason_mismatch,MinNTermAdd,minNTermSubtract,MinCTermAdd,minCTermSubtract
4844,24540,FLEM+15.995EPVTIPDVHGGSLQNAVR,FLEMEPVTIPDVHGGSLQNAVR,22,OtherLength,,0,0,0,0
7131,15548,QGEPLDDYVN+0.984AQGASLFSVTK,QGEPLDDYVNAQGASLFSVTK,21,OtherLength,,2,-1,0,0
7165,15548,QGEPLDDYVNAQGASLFSVTK,QGEPLDDYVNAQGASLFSVTK,21,OtherLength,,2,-1,0,0
7626,24669,GSAGGN+0.984AHSPLGVPGGGLPEHTFNLK,GSAGGNAHSPLGVPGGGLPEHTFNLK,26,OtherLength,,0,0,0,0
7666,19146,IHEGDITQILNSLLQGYDNKLR,IHEGDITQILNSLLQGYDNKLR,22,OtherLength,,0,0,0,0
